In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import cell2location

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42

In [ ]:
adata_st = sc.read_h5ad('/path/to/project/SingleCell/data/GSE284089_spatial.h5ad')
adata_st

In [ ]:
adata_ref = sc.read_h5ad('/path/to/project/SingleCell/data/GSE169396_scRNA.h5ad')
adata_ref

In [ ]:
adata_ref.obsm['X_umap'] = np.asarray(adata_ref.obsm['X_umap'])
adata_ref

In [ ]:

sc.settings.set_figure_params(dpi=100, dpi_save=300, figsize=(4, 4))
sc.pl.umap(adata_ref, color='cell_type')

In [ ]:
from cell2location.utils.filtering import filter_genes
selected = filter_genes(adata_ref, cell_count_cutoff=5, cell_percentage_cutoff2=0.03, nonz_mean_cutoff=1.12)

# filter the object
adata_ref = adata_ref[:, selected].copy()

In [ ]:
cell2location.models.RegressionModel.setup_anndata(adata=adata_ref,
                         batch_key='orig.ident', # 10X reaction / sample / batch
                         labels_key='cell_type' # cell type, covariate used for constructing signatures
                         #categorical_covariate_keys=['Method'] # multiplicative technical effects (platform, 3' vs 5', donor effect)
                       )

In [ ]:
# create the regression model
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref)

# view anndata_setup as a sanity check
mod.view_anndata_setup()

In [ ]:
adata_st1 = sc.read_h5ad('/path/to/project/SingleCell/cell2location/spatial_model/results/spatial_posterior_with_proportion.h5ad')
adata_st1

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 查看前 20 个基因
print(adata_st1.var_names[:20])

# 检查特定基因是否存在（例如 PDF 中提到的 CD68, RUNX2, SPP1）
genes_of_interest = ['CD68', 'RUNX2', 'SPP1', 'COL1A1', 'CD3D']
for gene in genes_of_interest:
    if gene in adata_st1.var_names:
        print(f"{gene} 存在于数据集中")
    else:
        print(f"警告：{gene} 不存在")

In [ ]:
# 提取一个基因的表达（返回的是稀疏矩阵，需要转为数组）
gene_name = 'CD68'
if gene_name in adata_st1.var_names:
    # 提取该基因在所有 Spot 中的表达量
    gene_expression = adata_st1[:, gene_name].X.toarray().flatten()
    
    # 将其存储到 adata.obs 中，方便后续绘图和统计分析
    adata_st1.obs[f'{gene_name}_expr'] = gene_expression
    
    # 查看统计信息
    print(adata_st1.obs[f'{gene_name}_expr'].describe())

In [ ]:
# 1. 将 spatial 转换为 numpy 数组
adata_st1.obsm['spatial'] = np.asarray(adata_st1.obsm['spatial'])

# 2. 检查一下数据类型
print(type(adata_st1.obsm['spatial'])) # 应该输出 <class 'numpy.ndarray'>

# 3. 重新运行你的绘图代码
sc.pl.spatial(
    adata_st1, 
    color='CD68_expr',       
    spot_size=15,            
    cmap='viridis',          
    title='CD68 Gene Expression'
)

In [ ]:
row_sum = abundance.sum(axis=1)

proportion = abundance.div(
    row_sum.replace(0, float("nan")),
    axis=0
)

In [ ]:
print(proportion.head())

print("\nRow sum:")
print(proportion.sum(axis=1).describe())